# The data model and dunder methods

Python’s data model is the layer that lets your own classes behave like built-in types. When you call `len(obj)`, compare two objects, use an object in a set, or iterate over it in a loop, Python is really looking for special methods such as `__len__`, `__eq__`, `__hash__`, and `__iter__`.

The power of dunder methods is that they make your abstractions feel natural. The danger is that incorrect implementations create subtle bugs, especially around hashing, equality, ordering, and context management.

As you move through these notebooks, think in terms of contracts: when Python calls a dunder method, what promise is your class making back to the rest of the language?

## Visual model

```text
your code -> len(x) / x == y / for item in x
             -> __len__ / __eq__ / __iter__
```

## How to use this notebook

Read the concept notes first, then run the code cells one at a time. After each run, change an input, prediction, or line of code and rerun it. Intermediate Python becomes easier when you treat every notebook as a place to test a mental model, not just a place to read finished answers.

---

**How to work through this.** Each task below is its own cell. Run them one at a
time and read the output before moving on; that is the whole advantage of a
notebook over a script. Where a cell asks for a prediction, write it before you
run anything. Being wrong on purpose in a place where it costs nothing is how
the correct model gets built.


---

# The concepts behind this exercise

Read this before the tasks. Every idea the tasks below use is explained here, so
you should not need to leave this notebook.

The code cells in this part are demonstrations rather than exercises. Run them,
change a value, run them again. That is the whole point of having them here
instead of in a document.

## Concept 1. `__repr__` and `__str__`

Implement `__repr__` on every class you write. It costs one line and it pays
back in every debugging session.

In [ ]:
class Point:
    def __init__(self, x: float, y: float) -> None:
        self.x, self.y = x, y

    def __repr__(self) -> str:
        return f"Point(x={self.x!r}, y={self.y!r})"    # for DEVELOPERS

    def __str__(self) -> str:
        return f"({self.x}, {self.y})"                  # for USERS

| | `__repr__` | `__str__` |
|---|---|---|
| Audience | Developers | End users |
| Goal | Unambiguous | Readable |
| Called by | REPL, `repr()`, containers, debuggers, logging | `print()`, `str()`, f-strings |
| Default | `<Point object at 0x7f...>` | Falls back to `__repr__` |
| Ideal | Valid Python that reconstructs the object | Whatever reads best |

Define `__repr__` always; define `__str__` only when the user-facing form
genuinely differs.

**The rule that matters:** a container's `str()` uses its elements' `repr()`.

```text
>>> print([Point(1, 2)])
[Point(x=1, y=2)]           # __repr__, not __str__
```


So a class with only `__str__` still prints as `<object at 0x...>` inside a
list, which is exactly when you most need to see it. And use `!r` inside your
repr: `f"{self.name!r}"` shows `'Ada'` rather than `Ada`, which distinguishes an
empty string from a missing value.

---

## Concept 2. `__eq__` and `__hash__` are a pair

In [ ]:
class Point:
    def __init__(self, x: float, y: float) -> None:
        self.x, self.y = x, y

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Point):
            return NotImplemented          # let the OTHER side try
        return (self.x, self.y) == (other.x, other.y)

    def __hash__(self) -> int:
        return hash((self.x, self.y))      # hash what __eq__ compares

**Three rules, and violating any one causes silent, hard-to-find bugs:**

1. **Equal objects must have equal hashes.** If not, a dict looks in the wrong
   bucket and your key becomes unreachable — present in memory, invisible to
   lookup.
2. **Only hash immutable state.** If a hashed attribute changes after insertion,
   the same unreachable-entry bug appears.
3. **Defining `__eq__` sets `__hash__` to `None`.** Python does this
   deliberately, because a default identity hash would violate rule 1. Your
   class becomes unhashable unless you define `__hash__` too:

In [ ]:
class Bad:
    def __eq__(self, other): return True

{Bad()}     # TypeError: unhashable type: 'Bad'

**Return `NotImplemented`, not `False`, for unknown types.** `NotImplemented`
tells Python "I do not know", so it tries `other.__eq__(self)` before falling
back to identity. Returning `False` claims authority you do not have and breaks
comparison with types written to interoperate with yours.

`@dataclass` generates all of this correctly (Module 11), which is the main
reason to use it.

---

## Concept 5. Iteration

In [ ]:
class Countdown:
    def __init__(self, start: int) -> None:
        self.start = start

    def __iter__(self):
        return CountdownIterator(self.start)      # a FRESH iterator each time


class CountdownIterator:
    def __init__(self, current: int) -> None:
        self.current = current

    def __iter__(self):
        return self                                # iterators return themselves

    def __next__(self) -> int:
        if self.current <= 0:
            raise StopIteration                    # the protocol's "done"
        self.current -= 1
        return self.current + 1

**Iterable versus iterator** is the distinction that matters:

| | Iterable | Iterator |
|---|---|---|
| Defines | `__iter__` | `__iter__` **and** `__next__` |
| Reusable | Yes — a fresh iterator each time | **No.** Once exhausted, done. |
| Examples | `list`, `dict`, `str`, `range` | `iter([])`, a generator, a file object |

Making a class its own iterator (returning `self` from `__iter__` and keeping the
position on the instance) means **two `for` loops over it cannot both work** —
the second sees an exhausted object. That is a real and confusing bug. Return a
fresh iterator, or write `__iter__` as a generator, which does it for you:

```text
    def __iter__(self):
        current = self.start          # local state -> fresh each call
        while current > 0:
            yield current
            current -= 1
```


Module 14 covers generators properly.

---

## Concept 6. Context managers

In [ ]:
class Timer:
    def __enter__(self) -> "Timer":
        self.start = time.perf_counter()
        return self                    # what `as x` binds

    def __exit__(self, exc_type, exc_value, traceback) -> bool:
        self.elapsed = time.perf_counter() - self.start
        return False                   # False/None: do NOT suppress exceptions

`__exit__` runs **whether or not** an exception occurred — that is the entire
point. Its three arguments are `None, None, None` on a clean exit.

**Returning `True` from `__exit__` swallows the exception.** Almost always
wrong. Do it only when suppression is the explicit purpose, as in
`contextlib.suppress`.

The concise form, which is what you will actually write (Module 15):

In [ ]:
from contextlib import contextmanager

@contextmanager
def timer():
    start = time.perf_counter()
    try:
        yield
    finally:                      # finally, not bare -- runs on exception too
        print(f"{time.perf_counter() - start:.3f}s")

---

---

# Now the exercise

You have everything you need. Work top to bottom, and where a cell asks for a
prediction, write it before you run anything.

## The concepts this exercise uses

These are the numbered sections of [the module README](../README.md). If a task below stops making sense, the section named next to it is the one to re-read.

- Section 1: `__repr__` and `__str__`
- Section 2: `__eq__` and `__hash__` are a pair
- Section 3: Ordering
- Section 4: The container protocols
- Section 5: Iteration
- Section 6: Context managers
- Section 7: Operators
- Section 8: `__call__`, `__bool__`, `__format__`
- Section 9: The whole map

> The teaching for this module currently lives in the README rather than in this notebook. Read it alongside these cells.

## Setup

Run this first. It is the imports and any shared values the tasks below need.

In [ ]:
from __future__ import annotations

from typing import Any


# --- broken 1 -----------------------------------------------------------------

---

## `UserA`

_UserA_

In [ ]:
class UserA:
    def __init__(self, user_id: int, name: str) -> None:
        self.id = user_id
        self.name = name

    def __eq__(self, other: object) -> bool:
        return isinstance(other, UserA) and self.id == other.id

---

## `UserB`

_UserB_

In [ ]:
class UserB:
    def __init__(self, user_id: int, name: str) -> None:
        self.id = user_id
        self.name = name

    def __eq__(self, other: object) -> bool:
        return isinstance(other, UserB) and self.id == other.id

    def __hash__(self) -> int:
        return hash((self.id, self.name))

---

## `Tag`

_Tag_

In [ ]:
class Tag:
    def __init__(self, label: str) -> None:
        self.label = label

    def __eq__(self, other: object) -> bool:
        return isinstance(other, Tag) and self.label == other.label

    def __hash__(self) -> int:
        return hash(self.label)

    def rename(self, new_label: str) -> None:
        self.label = new_label

---

## `Money`

_Money_

In [ ]:
class Money:
    def __init__(self, cents: int, currency: str = "USD") -> None:
        self.cents = cents
        self.currency = currency

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Money):
            return False                  # instead of NotImplemented
        return self.cents == other.cents and self.currency == other.currency

    def __hash__(self) -> int:
        return hash((self.cents, self.currency))

---

## `Cents`

A type that WANTS to compare equal to a Money of the same value.

In [ ]:
class Cents(int):
    """A type that WANTS to compare equal to a Money of the same value."""

    def __eq__(self, other: object) -> bool:
        if isinstance(other, Money):
            return int(self) == other.cents
        return int(self) == other

    def __hash__(self) -> int:
        return hash(int(self))

---

## `Point`

_Point_

In [ ]:
class Point:
    def __init__(self, x: float, y: float) -> None:
        self.x, self.y = x, y

    def __eq__(self, other: object) -> bool:
        return isinstance(other, Point) and (self.x, self.y) == (other.x, other.y)

    def __hash__(self) -> int:
        return 0            # "correct" but pathological. Why?

---

## `test_a_is_hashable`

_test a is hashable_

In [ ]:
def test_a_is_hashable() -> None:
    a = UserA(1, "Ada")
    assert {a: "x"}[UserA(1, "Ada")] == "x"

---

## `test_b_equal_objects_are_the_same_key`

_test b equal objects are the same key_

In [ ]:
def test_b_equal_objects_are_the_same_key() -> None:
    d = {UserB(1, "Ada"): "first"}
    d[UserB(1, "Ada Lovelace")] = "second"
    assert len(d) == 1, "two EQUAL objects created two dict entries"

---

## `test_tag_stays_findable_after_rename`

_test tag stays findable after rename_

In [ ]:
def test_tag_stays_findable_after_rename() -> None:
    t = Tag("python")
    s = {t}
    t.rename("py")
    assert t in s, "the object is in the set but cannot be found"

---

## `test_money_interoperates_with_cents`

_test money interoperates with cents_

In [ ]:
def test_money_interoperates_with_cents() -> None:
    assert Money(500) == Cents(500)
    assert Cents(500) == Money(500)

---

## `test_point_hash_is_distributed`

_test point hash is distributed_

In [ ]:
def test_point_hash_is_distributed() -> None:
    points = {Point(x, y) for x in range(100) for y in range(100)}
    assert len(points) == 10_000
    hashes = {hash(p) for p in points}
    assert len(hashes) > 1_000, (
        f"only {len(hashes)} distinct hashes for 10,000 objects: every lookup "
        "is a linear scan"
    )

---

## Run it

This is what running the original file did. Everything above must have been run first.

In [ ]:
if __name__ == "__main__":
    tests = [v for k, v in sorted(globals().items()) if k.startswith("test_")]
    for t in tests:
        try:
            t()
            print(f"  PASS  {t.__name__}")
        except Exception as exc:
            print(f"  FAIL  {t.__name__}: {type(exc).__name__}: {exc}")

---

## Before you move on

- [ ] Every cell above ran, in order, on a fresh kernel.
- [ ] You wrote a prediction before running, wherever one was asked for.
- [ ] You can say in one sentence what each task was actually testing.
- [ ] Anything that surprised you is written down in `PROGRESS.md`.

Compare against the worked answers in `../solutions/` only after your own
attempt runs.